# Lab 0 Task 0.2.1 — Transfer Learning from ImageNet
In this section, we use AlexNet on CIFAR-10 in two settings:
1. Fine-tuning the whole network
2. Feature extraction with pretrained weights

## Setup & Imports

In [1]:
import warnings
import copy
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision.models import alexnet, AlexNet_Weights
from typing import List, Dict, Tuple, Any
from utils import train, validate, fit

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Make results reproducible
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

2.11.0+cu130
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


## Load CIFAR-10

We apply ImageNet normalization and resize all images to 224x224 so they match AlexNet's expected input.

In [2]:
IMG_SIZE = 224
BATCH_SIZE = 256

imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)

transform_alexnet = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

train_dataset = torchvision.datasets.CIFAR10(
    root="../data", train=True, download=True, transform=transform_alexnet
)
test_dataset = torchvision.datasets.CIFAR10(
    root="../data", train=False, download=True, transform=transform_alexnet
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

classes: list[str] = train_dataset.classes
print("Classes:", classes)

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## AlexNet model
We replace the last classifier layer so the output dimension becomes 10.

In [3]:
def build_alexnet(pretrained=False, freeze_features=False):
    weights = AlexNet_Weights.DEFAULT if pretrained else None
    model = alexnet(weights=weights)

    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(in_features, 10)

    if freeze_features:
        for p in model.features.parameters():
            p.requires_grad = False
        for layer in model.classifier[:-1]:
            for p in layer.parameters():
                p.requires_grad = False

    return model

## Experiment A — Fine-tuning AlexNet on CIFAR-10
Here we train the whole network end-to-end.

In [5]:
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4

model_ft = build_alexnet(pretrained=False, freeze_features=False).to(device)
optimizer_ft = optim.Adam(model_ft.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="AlexNet Fine-tuning",
    tags=["Task 0.2.1", "Fine-tuning"],
    config={"pretrained": False, "freeze_features": False, "optimizer": "Adam",
            "lr": LEARNING_RATE, "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_ft = fit(
    model=model_ft, 
    optimizer=optimizer_ft,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=test_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: oscar-engelmark (d7047e-group12) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/5 | train loss 1.7031, train acc 37.65% | val loss 1.4278, val acc 48.67%
Epoch 2/5 | train loss 1.2843, train acc 53.52% | val loss 1.1293, val acc 59.74%
Epoch 3/5 | train loss 1.0567, train acc 62.50% | val loss 0.9992, val acc 64.61%
Epoch 4/5 | train loss 0.8948, train acc 68.34% | val loss 0.8708, val acc 70.05%
Epoch 5/5 | train loss 0.7779, train acc 72.65% | val loss 0.7621, val acc 73.89%


Training Accuracy,▁▄▆▇█
Training Loss,█▅▃▂▁
Validation Accuracy,▁▄▅▇█
Validation Loss,█▅▃▂▁
Training Accuracy,72.652
Training Loss,0.77785
Validation Accuracy,73.89
Validation Loss,0.76212



Restored best weights (val acc 73.89%)


## Experiment B — Feature extraction with pretrained AlexNet
Here we freeze the backbone and train only the final classifier layer.

In [ ]:
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3

model_fe = build_alexnet(pretrained=True, freeze_features=True).to(device)
optimizer_fe = optim.Adam([p for p in model_fe.parameters() if p.requires_grad], lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="AlexNet Feature Extraction",
    tags=["Task 0.2.1", "Feature Extraction"],
    config={"pretrained": True, "freeze_features": True, "optimizer": "Adam",
            "lr": LEARNING_RATE, "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_fe = fit(
    model=model_fe, 
    optimizer=optimizer_fe,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=test_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS
)

Epoch 1/5 | train loss 0.7383, train acc 74.08% | val loss 0.5556, val acc 80.82%
Epoch 2/5 | train loss 0.5984, train acc 79.02% | val loss 0.5117, val acc 81.91%
Epoch 3/5 | train loss 0.5662, train acc 79.97% | val loss 0.4967, val acc 82.87%
Epoch 4/5 | train loss 0.5491, train acc 80.68% | val loss 0.4928, val acc 82.88%


## Brief comparison
Fine-tuning updates the whole AlexNet, so the model can adapt to CIFAR-10 more strongly.
Feature extraction keeps pretrained layers frozen, so only the new classifier learns.
That is why fine-tuning usually performs better when the target dataset is different from ImageNet.

In [7]:
print("===== Final Comparison (Accuracy) =====")
print(f"Fine-tuning     best val acc: {max(history_ft['val_acc']):.2f}%")
print(f"Feature extract best val acc: {max(history_fe['val_acc']):.2f}%")

===== Final Comparison (Accuracy) =====
Fine-tuning     best val acc: 80.09%


NameError: name 'history_fe' is not defined